In [ ]:
# Llama-3.2 is gated. On Kaggle, store a secret named HF_TOKEN; elsewhere, export HF_TOKEN or use `hf auth login`.
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')

if token:
    login(token=token)
    print('Hugging Face login: OK')
else:
    print('No HF_TOKEN found; using the cached Hugging Face login.')

In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

import os
MODEL_ID = os.environ.get('A1_MODEL_ID', 'meta-llama/Llama-3.2-1B')
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

In [ ]:
# Question 4: SVD of the query projection (q_proj) weights in layers 0, 7 and 15
from pathlib import Path

LAYERS = [0, 7, 15]
TOP_K = 200            # visualise only the first 200 singular values
ENERGY_TARGET = 50.0   # % of cumulative energy to retain
OUT_DIR = Path(os.environ.get('A1_RESULT_ROOT', '/kaggle/working/Assignment1')) / 'q4'
OUT_DIR.mkdir(parents=True, exist_ok=True)

results = {}
for k in LAYERS:
    q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()
    U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)
    energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
    # smallest r such that the top-r singular values hold at least 50% of the energy (rank is 1-based)
    min_rank = int(torch.nonzero(energy_ratios >= ENERGY_TARGET)[0].item()) + 1
    results[k] = {'S': S.cpu(), 'energy': energy_ratios.cpu(), 'min_rank': min_rank, 'shape': tuple(q_weight.shape)}
    print(f'Layer {k:2d}: q_proj {tuple(q_weight.shape)}, full rank {len(S)}, '
          f'sigma_1 = {S[0]:.2f}, sigma_200 = {S[TOP_K - 1]:.2f}, '
          f'minimum rank for >= {ENERGY_TARGET:.0f}% energy = {min_rank} '
          f'(energy at rank 16/64/256: {energy_ratios[15]:.1f}% / {energy_ratios[63]:.1f}% / {energy_ratios[255]:.1f}%)')

In [ ]:
# Six charts: singular-value magnitudes (bar) and cumulative energy (line) for each layer
import json

fig, axes = plt.subplots(2, len(LAYERS), figsize=(6 * len(LAYERS), 9))
for col, k in enumerate(LAYERS):
    S = results[k]['S'][:TOP_K].numpy()
    energy = results[k]['energy'][:TOP_K].numpy()
    idx = range(1, TOP_K + 1)

    ax = axes[0, col]
    ax.bar(idx, S, width=1.0)
    ax.set_title(f'Layer {k} q_proj: singular value magnitudes')
    ax.set_xlabel('Singular Value Index')
    ax.set_ylabel('Singular Value Magnitude')
    ax.grid(True, alpha=0.3)

    ax = axes[1, col]
    ax.plot(idx, energy)
    ax.axhline(ENERGY_TARGET, color='gray', linestyle='--', linewidth=1)
    if results[k]['min_rank'] <= TOP_K:
        ax.axvline(results[k]['min_rank'], color='red', linestyle=':', linewidth=1,
                   label=f"min rank for 50% = {results[k]['min_rank']}")
        ax.legend()
    ax.set_title(f'Layer {k} q_proj: cumulative energy (%)')
    ax.set_xlabel('Target Rank')
    ax.set_ylabel('Cumulative Energy (%)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'q4_svd_q_proj.png', dpi=150)
plt.show()

summary = {f'layer_{k}': {'min_rank_50pct': results[k]['min_rank'],
                          'energy_at_rank': {r: round(float(results[k]['energy'][r - 1]), 2) for r in (1, 16, 64, 128, 200, 256, 512)},
                          'top5_singular_values': [round(float(x), 3) for x in results[k]['S'][:5]]}
           for k in LAYERS}
(OUT_DIR / 'q4_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps({k: v['min_rank_50pct'] for k, v in summary.items()}))